# 01 · Retrieval baseline

This notebook builds the lexical (BM25) retrieval baseline directly on the
`ragops_lab` package — the same code paths used by the CLI and API. We:

1. ingest the sample corpus into reusable chunks,
2. index the chunks with the BM25 retriever, and
3. inspect the ranked results for a sample query.

Everything here is package code, not notebook-only logic, so the behaviour you
see matches production.

In [1]:
from pathlib import Path

import pandas as pd

from ragops_lab.ingestion import ChunkingConfig, ingest_directory, load_chunks_jsonl
from ragops_lab.retrieval import BM25Retriever

pd.set_option("display.max_colwidth", 90)
DATA = Path("../data")

## 1. Ingest the sample corpus

`ingest_directory` discovers documents, chunks them with the configured window,
and persists the chunks as JSONL so retrieval and the CLI can reload them.

In [2]:
chunks_path = DATA / "processed" / "chunks.jsonl"
ingest_directory(
    DATA / "sample_documents",
    chunks_path,
    ChunkingConfig(chunk_size=220, overlap=20),
)
chunks = load_chunks_jsonl(chunks_path)

pd.DataFrame(
    {
        "chunk_id": [c.chunk_id for c in chunks],
        "tokens": [len(c.text.split()) for c in chunks],
        "preview": [c.text[:80] for c in chunks],
    }
)

,chunk_id,tokens,preview
0,apollo-program:0,39,The Apollo program landed humans on the Moon. Apollo 11 was the first mission to
1,apollo-program:1,5,ins remained in lunar orbit.
2,rag-evaluation:0,26,RAG evaluation should measure retrieval quality and answer quality. Useful metri
3,readme:0,20,# Sample documents\n\nPlace small public-domain or synthetic documents here for lo


## 2. Index with BM25 and run a query

The retriever scores every chunk for the query terms and returns ranked
`RetrievalResult` objects carrying the score, rank, method, and matched terms.

In [3]:
retriever = BM25Retriever(chunks)
results = retriever.search("Which Apollo mission first landed on the Moon?", top_k=3)

pd.DataFrame(
    {
        "rank": [r.rank for r in results],
        "chunk_id": [r.chunk.chunk_id for r in results],
        "score": [round(r.score, 3) for r in results],
        "matched_terms": [", ".join(r.matched_terms) for r in results],
        "preview": [r.chunk.text[:70] for r in results],
    }
)

,rank,chunk_id,score,matched_terms,preview
0,1,apollo-program:0,9.25,"apollo, first, landed, mission, moon, on, the",The Apollo program landed humans on the Moon. Apollo 11 was the first


## 3. Inspect the top hit

The matched terms explain *why* a chunk ranked first — useful when debugging
retrieval quality before generation ever runs.

In [4]:
top = results[0]
print(f"chunk_id     : {top.chunk.chunk_id}")
print(f"method       : {top.retrieval_method}")
print(f"score        : {top.score:.3f}")
print(f"matched terms: {top.matched_terms}")
print()
print(top.chunk.text)

chunk_id     : apollo-program:0
method       : lexical
score        : 9.250
matched terms: ['apollo', 'first', 'landed', 'mission', 'moon', 'on', 'the']

The Apollo program landed humans on the Moon. Apollo 11 was the first mission to land astronauts on the lunar surface in July 1969. Neil Armstrong and Buzz Aldrin walked on the Moon while Michael Collins remained in luna


Next: [`02_retrieval_strategies`](02_retrieval_strategies.ipynb) compares this
baseline against vector and hybrid retrieval and scores them on a golden set.